# THGS Pipeline — Ramen Scene
**Training-Free Hierarchical Scene Understanding for Gaussian Splatting with Superpoint Graphs**

이 노트북은 THGS 논문의 전체 파이프라인을 **LERF-OVS `ramen` 씬** 대상으로 Colab 환경에서 실행합니다.

| 단계 | 내용 |
|------|------|
| Cell 0 | 환경 설정 & 의존성 설치 |
| Cell 1 | 데이터셋 다운로드 & 2DGS 장면 준비 (ramen) |
| Cell 2 | 언어 피처 생성 (SAM + CLIP) |
| Cell 3 | THGS 파이프라인 (인접 그래프 → 엣지 가중치 → 슈퍼포인트 분할 → 계층적 병합) |
| Cell 4 | 평가 (Open-vocabulary Segmentation) |
| Cell 5 | 시각화 |

> **Ramen 씬 특징**
> - 라면 그릇 + 식재료(계란, 김, 가마보코, 차슈, 면, 젓가락, 숟가락, 그릇 등)
> - **Part-level 쿼리가 많은 씬** (Limitation 가설 H2 검증에 적합)
> - 평가 프롬프트 예시: `bowl`, `chopsticks`, `egg`, `glass of water`, `kamaboko`, `napkin`, `nori`, `pork belly`, `sake cup`, `spoon`, `wavy noodles in bowl`

---
## Cell 0. 환경 설정 & 의존성 설치

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Google Drive 마운트 (모든 캐시 저장/복원에 사용)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/THGS_cache"
import os
os.makedirs(DRIVE_CACHE, exist_ok=True)
print(f"Drive 캐시: {DRIVE_CACHE}")
print(f"  기존 캐시: {os.listdir(DRIVE_CACHE) if os.listdir(DRIVE_CACHE) else '없음'}")

In [ ]:
# GitHub 레포에서 THGS 클론 + submodule 무결성 검증/복구
import os

REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"

# 1) 레포 없으면 --recursive 최초 clone
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)

# 2) 원격 변경사항 pull (conflict 방지용 autostash)
!git pull --rebase --autostash 2>&1 | tail -3

# 3) submodule URL 동기화 (.gitmodules 변경 반영)
!git submodule sync --recursive

# 4) 강제 재초기화 + fetch (--force: working tree 꼬여도 덮어씀)
!git submodule update --init --recursive --force 2>&1 | tail -20

# 5) 핵심 소스 파일 존재 여부로 실제 무결성 검증
REQUIRED_FILES = [
    "submodules/simple-knn/simple_knn.cu",
    "submodules/simple-knn/spatial.cu",
    "submodules/simple-knn/setup.py",
    "submodules/diff-surfel-rasterization/setup.py",
    "ext/spt/dependencies/FRNN/setup.py",
    "ext/spt/dependencies/parallel_cut_pursuit",
    "ext/spt/dependencies/grid_graph",
]

def _submodule_root(path):
    '''파일 경로 → submodule 루트 디렉터리'''
    parts = path.split('/')
    if parts[0] == "submodules":
        return "/".join(parts[:2])          # submodules/<name>
    if parts[0] == "ext" and parts[1] == "spt":
        return "/".join(parts[:4])          # ext/spt/dependencies/<name>
    return parts[0]

def _check_missing():
    return [p for p in REQUIRED_FILES if not os.path.exists(p)]

missing = _check_missing()

# 6) 누락된 게 있으면: 해당 submodule 디렉터리 통째 삭제 후 재클론
if missing:
    print(f"[REPAIR] 누락된 파일/폴더: {missing}")
    broken_roots = sorted({_submodule_root(p) for p in missing})
    for root in broken_roots:
        if os.path.exists(root):
            print(f"  rm -rf {root}")
            !rm -rf {root}
    print("  git submodule update --init --recursive 재실행...")
    !git submodule update --init --recursive 2>&1 | tail -10
    missing = _check_missing()
    if missing:
        raise RuntimeError(
            f"submodule 복구 실패: {missing}\n"
            "원인 후보: 네트워크 일시 장애(특히 gitlab.inria.fr), 권한, Colab DNS.\n"
            "해결: 잠시 후 이 셀을 다시 실행하거나 런타임을 해제 후 재시도."
        )

# 7) 최종 검증 리포트
print()
print("=" * 60)
print("submodule 무결성 체크")
print("=" * 60)
for p in REQUIRED_FILES:
    status = "OK " if os.path.exists(p) else "MISS"
    print(f"  [{status}] {p}")

print()
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
# 의존성 설치

# 0) NumPy 2.x와 호환되는 opencv 설치를 위해 먼저 업그레이드
!pip install -q --upgrade opencv-python

# 1) 기본 pip 패키지
!pip install -q plyfile==0.8.1 trimesh kiui pymeshlab open3d scipy \
    omegaconf open_clip_torch transformations transformers yapf \
    pycocotools mediapy lpips scikit-image einops dearpygui \
    h5py colorhash seaborn pyrootutils hydra-core hydra-colorlog \
    hydra-submitit-launcher numba pytorch-lightning rich \
    ipyfilechooser natsort gdown ninja

# 2) PyTorch Geometric (SPT 라이브러리가 PyG 2.3.0 기준 → 반드시 고정)
import torch
CUDA_VERSION = torch.version.cuda.replace(".", "")
TORCH_VERSION = torch.__version__.split("+")[0]
print(f"PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}")

!pip install -q torch_geometric==2.3.0
!pip install -q pyg_lib torch_scatter torch_cluster \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html \
    || pip install -q torch_scatter torch_cluster

# 버전 확인
import torch_geometric
print(f"torch_geometric: {torch_geometric.__version__} (2.3.0 필수)")

In [ ]:
# 3) CUDA 서브모듈 빌드
import torch, os

# 0) 사전 검증: 소스 파일이 실제 존재하는지 확인 (submodule 깨졌으면 여기서 바로 중단)
required_src = [
    "submodules/simple-knn/simple_knn.cu",
    "submodules/simple-knn/spatial.cu",
    "submodules/simple-knn/setup.py",
    "submodules/diff-surfel-rasterization/setup.py",
]
_missing = [p for p in required_src if not os.path.exists(p)]
assert not _missing, (
    f"서브모듈 소스 누락: {_missing}\n"
    f"→ Cell 4(클론 셀)를 다시 실행하세요. Cell 4의 자동 복구 로직이 처리합니다."
)

# CUDA_HOME 설정 & GPU 아키텍처 자동 감지
CUDA_HOME = os.environ.get("CUDA_HOME", "/usr/local/cuda")
os.environ["CUDA_HOME"] = CUDA_HOME
cc = torch.cuda.get_device_capability()
TORCH_CUDA_ARCH = f"{cc[0]}.{cc[1]}"
os.environ["TORCH_CUDA_ARCH_LIST"] = TORCH_CUDA_ARCH
print(f"CUDA_HOME: {CUDA_HOME}")
print(f"GPU arch: {TORCH_CUDA_ARCH} ({torch.cuda.get_device_name(0)})")

# simple-knn 소스 패치 (CUDA 12.x / Blackwell 호환)
# 1) FLT_MAX 미정의 → #include <cfloat> 추가, 중복 #define __CUDACC__ 제거
!sed -i 's|#include "simple_knn.h"|#include "simple_knn.h"\n#include <cfloat>|' submodules/simple-knn/simple_knn.cu
!sed -i '/#define __CUDACC__/d' submodules/simple-knn/simple_knn.cu
# 2) deprecated data<float>() → data_ptr<float>()
!sed -i 's/\.data<float>()/\.data_ptr<float>()/g' submodules/simple-knn/spatial.cu

# 빌드
!pip install submodules/diff-surfel-rasterization 2>&1 | tail -3
!pip install submodules/simple-knn 2>&1 | tail -3

# 빌드 검증
try:
    from diff_surfel_rasterization import GaussianRasterizer
    print("[OK] diff-surfel-rasterization")
except Exception as e:
    print(f"[FAIL] diff-surfel-rasterization: {e}")

try:
    import simple_knn
    print("[OK] simple-knn")
except Exception as e:
    print(f"[FAIL] simple-knn: {e}")

In [ ]:
# 4) SPT 의존성 빌드
!pip install -q ext/spt/dependencies/FRNN
!pip install -q ext/spt/dependencies/FRNN/external/prefix_sum

# grid_graph + parallel_cut_pursuit C++ 확장 컴파일
!python scripts/setup_dependencies.py build_ext

In [ ]:
# 5) 추가 의존성
!pip install -q git+https://github.com/facebookresearch/pytorch3d.git
!pip install -q git+https://github.com/drprojects/point_geometric_features.git
!pip install -q git+https://github.com/minghanqin/segment-anything-langsplat.git

In [ ]:
# 설치 검증
import torch
import open_clip
import torch_geometric
from diff_surfel_rasterization import GaussianRasterizer

print("-- 설치 검증 완료 --")
print(f"  torch: {torch.__version__}")
print(f"  torch_geometric: {torch_geometric.__version__}")
print(f"  open_clip: {open_clip.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

---
## Cell 1. 데이터셋 다운로드 & 2DGS 베이스 준비

저자 제공 Pre-trained는 `pretrained/` 폴더에 보관 (참고용).  
`output/`에는 2DGS 베이스 파일만 복사하고, 파이프라인 결과는 직접 생성합니다.

In [ ]:
# ============================================================
# 설정: 사용할 데이터셋과 장면 선택 (Ramen 씬 전용)
# ============================================================
DATASET = "lerf"
CONFIG_FILE = f"configs/{DATASET}.yml"
SCENES = ["ramen"]                                   # ← 이 노트북은 ramen 씬만 실행
# 다른 LERF 씬: ["figurines", "teatime", "waldo_kitchen"]
# 3DOVS: ["bed", "bench", "lawn", "room", "sofa"]

print(f"Dataset: {DATASET}")
print(f"Config: {CONFIG_FILE}")
print(f"Scenes: {SCENES}")

In [ ]:
# 데이터셋 다운로드 & 2DGS 베이스 준비
import os
import gdown
import zipfile
import glob
import shutil

# ---- Google Drive IDs ----
LERF_DATA_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"          # LERF-OVS zip (from LangSplat)
LERF_DATA_ZIP = "/content/lerf_ovs.zip"
DOVS3_FOLDER_ID = "1kdV14Gu5nZX6WOPbccG7t7obP_aXkOuC"        # 3DOVS folder
PRETRAINED_FOLDER_ID = "1b3bXy8XENhvpWh4nLzu06UPBZEZNX6fG"    # Pre-trained scenes folder

# ============================================================
# 디렉토리 구조:
#   data/lerf-ovs/<scene>/images/, sparse/0/, (language_features/)
#   data/lerf-ovs/label/<scene>/   ← GT 마스크 (평가용)
#   pretrained/lerf/<scene>/       ← 저자 제공 원본 (sai_nag.pt 포함, 참고용)
#   output/lerf/<scene>/           ← 내 작업 공간 (2DGS 베이스 + 직접 생성한 파이프라인 결과)
# ============================================================

# 2DGS 베이스 파일 (저자 제공 → output으로 복사, sai_nag.pt 등은 제외)
BASE_FILES = ["cfg_args", "cameras.json", "input.ply"]
BASE_DIRS = ["point_cloud"]

def copy_base_to_output(pretrained_scene_dir, output_scene_dir):
    """저자 제공에서 2DGS 베이스 파일만 output으로 복사 (sai_nag.pt 등 제외)"""
    os.makedirs(output_scene_dir, exist_ok=True)
    for f in BASE_FILES:
        src = os.path.join(pretrained_scene_dir, f)
        dst = os.path.join(output_scene_dir, f)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    for d in BASE_DIRS:
        src = os.path.join(pretrained_scene_dir, d)
        dst = os.path.join(output_scene_dir, d)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)

def download_and_extract_pretrained(pretrained_dir):
    """gdown으로 저자 제공 zip 다운로드 후 해제"""
    TMP_DIR = "pretrained/_tmp"
    gdown.download_folder(id=PRETRAINED_FOLDER_ID, output=TMP_DIR, quiet=False)
    for d in [TMP_DIR] + glob.glob(f"{TMP_DIR}/*/"):
        for zf in glob.glob(f"{d}/*.zip"):
            name = os.path.splitext(os.path.basename(zf))[0]
            print(f"  {name}.zip 해제 → {pretrained_dir}/")
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(pretrained_dir)
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    # 혹시 pretrained/ 루트에 남은 zip도 해제
    for zf in glob.glob(f"{os.path.dirname(pretrained_dir)}/*.zip"):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(pretrained_dir)
        os.remove(zf)

if DATASET == "lerf":
    DATA_DIR = "data/lerf-ovs"
    PRETRAINED_DIR = "pretrained/lerf"
    OUTPUT_DIR = "output/lerf"

    # [1/3] LERF-OVS 데이터셋
    # SCENES의 모든 장면이 있는지 확인
    all_data_exist = all(os.path.exists(f"{DATA_DIR}/{sc}") for sc in SCENES)
    if not all_data_exist:
        print("[1/3] LERF-OVS 데이터셋 다운로드 중...")
        gdown.download(id=LERF_DATA_ID, output=LERF_DATA_ZIP, quiet=False)
        print("압축 해제 중...")
        with zipfile.ZipFile(LERF_DATA_ZIP, 'r') as z:
            z.extractall("data/")
        os.remove(LERF_DATA_ZIP)
        if not os.path.exists(DATA_DIR) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_DIR)
            print(f"  심볼릭 링크: {DATA_DIR} -> data/lerf_ovs")
        print(f"  완료: {os.listdir(DATA_DIR)}")
    else:
        print(f"[1/3] LERF-OVS 이미 존재: {os.listdir(DATA_DIR)}")

    # [2/3] 저자 제공 Pre-trained → pretrained/lerf/
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    all_pretrained_exist = all(
        os.path.exists(f"{PRETRAINED_DIR}/{sc}/point_cloud") for sc in SCENES
    )
    if not all_pretrained_exist:
        print(f"\n[2/3] 저자 제공 Pre-trained 다운로드 → {PRETRAINED_DIR}/")
        download_and_extract_pretrained(PRETRAINED_DIR)
        print(f"  완료: {os.listdir(PRETRAINED_DIR)}")
    else:
        print(f"[2/3] 저자 Pre-trained 이미 존재: {os.listdir(PRETRAINED_DIR)}")

    # [3/3] 2DGS 베이스만 output으로 복사 (sai_nag.pt 제외)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for scene in SCENES:
        pretrained_scene = f"{PRETRAINED_DIR}/{scene}"
        output_scene = f"{OUTPUT_DIR}/{scene}"
        ply_check = f"{output_scene}/point_cloud/iteration_30000/point_cloud.ply"
        if not os.path.exists(ply_check):
            if os.path.exists(pretrained_scene):
                print(f"[3/3] {scene}: 2DGS 베이스 → output/ 복사 (sai_nag.pt 제외)")
                copy_base_to_output(pretrained_scene, output_scene)
            else:
                print(f"[3/3] {scene}: pretrained 없음 - 직접 2DGS 학습 필요")
        else:
            print(f"[3/3] {scene}: output에 2DGS 베이스 이미 존재")

elif DATASET == "3dovs":
    DATA_DIR = "data/3DOVS"
    PRETRAINED_DIR = "pretrained/3dovs"
    OUTPUT_DIR = "output/3dovs"

    all_data_exist = all(os.path.exists(f"{DATA_DIR}/{sc}") for sc in SCENES)
    if not all_data_exist:
        print("[1/3] 3DOVS 데이터셋 다운로드 중...")
        os.makedirs(DATA_DIR, exist_ok=True)
        gdown.download_folder(id=DOVS3_FOLDER_ID, output=DATA_DIR, quiet=False)
    else:
        print(f"[1/3] 3DOVS 이미 존재: {os.listdir(DATA_DIR)}")

    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    all_pretrained_exist = all(
        os.path.exists(f"{PRETRAINED_DIR}/{sc}/point_cloud") for sc in SCENES
    )
    if not all_pretrained_exist:
        print(f"\n[2/3] 저자 제공 Pre-trained 다운로드 → {PRETRAINED_DIR}/")
        download_and_extract_pretrained(PRETRAINED_DIR)
    else:
        print(f"[2/3] 저자 Pre-trained 이미 존재: {os.listdir(PRETRAINED_DIR)}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for scene in SCENES:
        pretrained_scene = f"{PRETRAINED_DIR}/{scene}"
        output_scene = f"{OUTPUT_DIR}/{scene}"
        ply_check = f"{output_scene}/point_cloud/iteration_30000/point_cloud.ply"
        if not os.path.exists(ply_check) and os.path.exists(pretrained_scene):
            print(f"[3/3] {scene}: 2DGS 베이스 → output/ 복사")
            copy_base_to_output(pretrained_scene, output_scene)
        else:
            print(f"[3/3] {scene}: output에 2DGS 베이스 이미 존재")

In [ ]:
# 디렉토리 구조 확인
print("=" * 60)
print("디렉토리 구조 확인")
print("=" * 60)

from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

for scene in SCENES:
    print(f"\n--- {scene} ---")
    data_path = f"{DATA_PATH}/{scene}"
    pretrained_path = f"pretrained/{SAVE_FOLDER}/{scene}"
    output_path = f"output/{SAVE_FOLDER}/{scene}"

    # 데이터 확인
    if os.path.exists(data_path):
        contents = os.listdir(data_path)
        print(f"  data:       {data_path} → {contents}")
    else:
        print(f"  data:       NOT FOUND at {data_path}")

    # pretrained 확인
    if os.path.exists(pretrained_path):
        contents = os.listdir(pretrained_path)
        print(f"  pretrained: {pretrained_path} → {contents}")
    else:
        print(f"  pretrained: NOT FOUND at {pretrained_path}")

    # output 확인 (sai_nag.pt 없는 게 정상 - 직접 생성해야 함)
    if os.path.exists(output_path):
        contents = os.listdir(output_path)
        ply_exists = os.path.exists(f"{output_path}/point_cloud/iteration_30000/point_cloud.ply")
        sai_exists = os.path.exists(f"{output_path}/sai_nag.pt")
        print(f"  output:     {output_path} → {contents}")
        print(f"              point_cloud.ply: {'OK' if ply_exists else 'MISSING'}")
        print(f"              sai_nag.pt: {'있음 (직접 생성 또는 Drive 복원)' if sai_exists else '없음 (Cell 3에서 생성)'}")
    else:
        print(f"  output:     NOT FOUND at {output_path}")

In [ ]:
# Drive 캐시에서 language_features + 파이프라인 산출물 복원
# 이전 런타임에서 백업해둔 게 있으면 자동으로 가져옵니다.
import shutil
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

# --- language_features 복원 ---
skip_encoding = True
for scene in SCENES:
    local_feat = f"{DATA_PATH}/{scene}/language_features"
    drive_feat = f"{DRIVE_CACHE}/{scene}_language_features"

    if os.path.exists(local_feat) and len(os.listdir(local_feat)) > 0:
        print(f"[feat] {scene}: 로컬에 존재")
    elif os.path.exists(drive_feat):
        print(f"[feat] {scene}: Drive에서 복원 중...")
        shutil.copytree(drive_feat, local_feat)
        print(f"  완료 ({len(os.listdir(local_feat))} files)")
    else:
        print(f"[feat] {scene}: 없음 - Cell 2에서 생성 필요")
        skip_encoding = False

# --- 파이프라인 산출물 복원 (sai_nag.pt 등) ---
PIPELINE_FILES = ["neighbor.pt", "neighbor_new.pt", "nag-l1.pt", "sai_nag.pt"]
skip_pipeline = True
for scene in SCENES:
    model_path = f"output/{SAVE_FOLDER}/{scene}"
    drive_model = f"{DRIVE_CACHE}/{scene}_model"
    os.makedirs(model_path, exist_ok=True)

    if os.path.exists(f"{model_path}/sai_nag.pt"):
        print(f"[model] {scene}: sai_nag.pt 로컬에 존재")
    elif os.path.exists(f"{drive_model}/sai_nag.pt"):
        print(f"[model] {scene}: Drive에서 산출물 복원 중...")
        for f in PIPELINE_FILES:
            src = f"{drive_model}/{f}"
            dst = f"{model_path}/{f}"
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  {f} 복원")
        print(f"  완료")
    else:
        print(f"[model] {scene}: sai_nag.pt 없음 - Cell 3에서 생성 필요")
        skip_pipeline = False

if skip_encoding and skip_pipeline:
    print("\n==> 모든 캐시 복원 완료. Cell 2, 3 스킵 → Cell 4(평가)로 바로 이동 가능")

---
## Cell 2. 언어 피처 생성 (SAM + CLIP)

각 이미지에서 SAM으로 multi-scale 마스크를 추출하고, CLIP으로 인코딩합니다.
- 이미지당 수십 초 소요, 장면당 수십 분~1시간
- **Google Drive에 백업 → 런타임 재시작 시 복원하여 스킵**

In [ ]:
# SAM 체크포인트 다운로드 (~2.5GB) - 피처 생성이 필요할 때만
SAM_CKPT = "ckpts/sam_vit_h_4b8939.pth"
if not skip_encoding and not os.path.exists(SAM_CKPT):
    os.makedirs("ckpts", exist_ok=True)
    !wget -q --show-progress -O {SAM_CKPT} \
        https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("SAM 체크포인트 다운로드 완료")
elif skip_encoding:
    print("스킵: language_features 이미 존재")
else:
    print("SAM 체크포인트 이미 존재")

In [ ]:
# 언어 피처 생성
if not skip_encoding:
    for scene in SCENES:
        feat_path = f"{DATA_PATH}/{scene}/language_features"
        if os.path.exists(feat_path) and len(os.listdir(feat_path)) > 0:
            print(f"{scene}: 이미 존재, 스킵")
            continue
        print(f"\n{'='*50}")
        print(f"{scene}: 언어 피처 생성 시작")
        print(f"{'='*50}")
        !python scripts/image_encoding.py \
            --source_path {DATA_PATH}/{scene} \
            --sam_ckpt_path {SAM_CKPT}
else:
    print("스킵: 모든 장면에 language_features가 이미 있습니다.")

In [ ]:
# language_features → Google Drive 백업
import shutil

for scene in SCENES:
    local_feat = f"{DATA_PATH}/{scene}/language_features"
    drive_feat = f"{DRIVE_CACHE}/{scene}_language_features"

    if os.path.exists(local_feat) and not os.path.exists(drive_feat):
        n_files = len(os.listdir(local_feat))
        print(f"{scene}: Drive에 백업 중 ({n_files} files)...")
        shutil.copytree(local_feat, drive_feat)
        print(f"  완료: {drive_feat}")
    elif os.path.exists(drive_feat):
        print(f"{scene}: Drive에 이미 백업됨")
    else:
        print(f"{scene}: language_features 없음 - 위 셀을 먼저 실행하세요")

---
## Cell 3. THGS 파이프라인 실행

핵심 4단계를 순서대로 실행합니다:
1. **인접 그래프 구축** - FRNN K-NN → `neighbor.pt`
2. **SAM 기반 엣지 가중치 조정** → `neighbor_new.pt`
3. **슈퍼포인트 분할** - Parallel Cut Pursuit → `nag-l1.pt`
4. **계층적 병합 + 피처 재투영** → `sai_nag.pt`

In [ ]:
# 전체 파이프라인 실행 (run.sh)
# sai_nag.pt가 없는 장면만 실행
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder

scenes_to_run = []
for scene in SCENES:
    sai_path = f"output/{SAVE_FOLDER}/{scene}/sai_nag.pt"
    if os.path.exists(sai_path):
        print(f"{scene}: sai_nag.pt 이미 존재 → 스킵")
    else:
        scenes_to_run.append(scene)

if scenes_to_run:
    scenes_arg = " ".join(scenes_to_run)
    print(f"\n실행: bash scripts/run.sh {CONFIG_FILE} {scenes_arg}")
    print("=" * 60)
    !bash scripts/run.sh {CONFIG_FILE} {scenes_arg}
else:
    scenes_arg = " ".join(SCENES)
    print("\n모든 장면에 sai_nag.pt가 존재합니다. 파이프라인 스킵.")

In [ ]:
# (선택) 개별 단계 실행 - 디버깅 시 사용
# 위 셀을 실행했으면 이 셀은 스킵

RUN_INDIVIDUAL = False

if RUN_INDIVIDUAL:
    _scenes_arg = " ".join(SCENES)

    print("[Step 3-1] 인접 그래프 구축 (FRNN K-NN)")
    print("=" * 60)
    !python scripts/launcher.py -f sp_partition.py -cf {CONFIG_FILE} -sc {_scenes_arg}

    print("\n[Step 3-2] SAM 기반 엣지 가중치 조정")
    print("=" * 60)
    !python scripts/launcher.py -f graph_weight.py -cf {CONFIG_FILE} -sc {_scenes_arg}

    print("\n[Step 3-3] 슈퍼포인트 분할 (Parallel Cut Pursuit)")
    print("=" * 60)
    !python scripts/launcher.py -f sp_partition.py -cf {CONFIG_FILE} -sc {_scenes_arg} -k

    print("\n[Step 3-4] 계층적 병합 + CLIP 피처 재투영")
    print("=" * 60)
    !python scripts/launcher.py -f merge_proj.py -cf {CONFIG_FILE} -sc {_scenes_arg}

In [ ]:
# 파이프라인 산출물 확인
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder

print("파이프라인 산출물 확인:")
print("=" * 60)
for scene in SCENES:
    model_path = f"output/{SAVE_FOLDER}/{scene}"
    print(f"\n--- {scene} ---")

    files_to_check = [
        ("neighbor.pt",     "Step 3-1: K-NN 인접 그래프"),
        ("neighbor_new.pt", "Step 3-2: 재가중된 그래프"),
        ("nag-l1.pt",       "Step 3-3: 슈퍼포인트 분할"),
        ("sai_nag.pt",      "Step 3-4: 최종 계층적 그래프 + 피처"),
    ]
    for fname, desc in files_to_check:
        fpath = os.path.join(model_path, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f"  [OK] {fname} ({size_mb:.1f} MB) - {desc}")
        else:
            print(f"  [X]  {fname} - {desc}")

In [ ]:
# 파이프라인 산출물 → Google Drive 백업
import shutil

PIPELINE_FILES = ["neighbor.pt", "neighbor_new.pt", "nag-l1.pt", "sai_nag.pt"]

for scene in SCENES:
    model_path = f"output/{SAVE_FOLDER}/{scene}"
    drive_model = f"{DRIVE_CACHE}/{scene}_model"
    os.makedirs(drive_model, exist_ok=True)

    for f in PIPELINE_FILES:
        src = f"{model_path}/{f}"
        dst = f"{drive_model}/{f}"
        if os.path.exists(src) and not os.path.exists(dst):
            size_mb = os.path.getsize(src) / 1024 / 1024
            print(f"{scene}: {f} ({size_mb:.1f} MB) → Drive 백업 중...")
            shutil.copy2(src, dst)
        elif os.path.exists(dst):
            pass  # 이미 백업됨
        else:
            print(f"{scene}: {f} 없음 - 파이프라인을 먼저 실행하세요")

    # sai_nag.pt 백업 확인
    if os.path.exists(f"{drive_model}/sai_nag.pt"):
        print(f"{scene}: Drive 백업 완료")
    else:
        print(f"{scene}: sai_nag.pt 백업 실패 - 확인 필요")

---
## Cell 4. 평가 (Open-vocabulary Segmentation)

- `test_lerf.py`: 텍스트 쿼리로 2D 마스크 예측
- `eval_seg.py`: 예측 마스크 vs GT 비교 (IoU 등)

In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.load(CONFIG_FILE)
DATA_PATH = cfg.dataset.data_path
SAVE_FOLDER = cfg.dataset.save_folder

PRED_PATH = f"output/render/{SAVE_FOLDER}"
GT_PATH = f"{DATA_PATH}/label"
os.makedirs(PRED_PATH, exist_ok=True)

# 각 장면에 대해 마스크 예측
for scene in SCENES:
    print(f"\n{'='*50}")
    print(f"{scene}: 마스크 예측 중...")
    print(f"{'='*50}")
    !python test_lerf.py \
        -s {DATA_PATH}/{scene} \
        -m output/{SAVE_FOLDER}/{scene} \
        --path_pred {PRED_PATH}

In [ ]:
# 정량 평가
scenes_list = " ".join(SCENES)
print("정량 평가:")
print("=" * 60)
!python scripts/eval_seg.py \
    --dataset {DATASET} \
    --scene_list {scenes_list} \
    --path_pred {PRED_PATH} \
    --path_gt {GT_PATH}

---
## Cell 5. 시각화

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def visualize_predictions(pred_path, scene_name, max_prompts=6):
    """예측 마스크와 GT를 나란히 시각화"""
    scene_dir = Path(pred_path) / scene_name
    if not scene_dir.exists():
        print(f"{scene_name}: 예측 결과 없음")
        return

    image_dirs = sorted([d for d in scene_dir.iterdir() if d.is_dir()])

    for img_dir in image_dirs:
        pred_files = sorted([f for f in img_dir.glob("*.png") if "_gt" not in f.name])
        gt_files = sorted(img_dir.glob("*_gt.png"))

        n = min(len(pred_files), max_prompts)
        if n == 0:
            continue

        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1:
            axes = axes.reshape(2, 1)

        fig.suptitle(f"{scene_name} / {img_dir.name}", fontsize=14)

        for i in range(n):
            pred = cv2.imread(str(pred_files[i]), cv2.IMREAD_GRAYSCALE)
            prompt_name = pred_files[i].stem

            axes[0, i].imshow(pred, cmap="gray")
            axes[0, i].set_title(f"Pred: {prompt_name}", fontsize=10)
            axes[0, i].axis("off")

            gt_file = img_dir / f"{prompt_name}_gt.png"
            if gt_file.exists():
                gt = cv2.imread(str(gt_file), cv2.IMREAD_GRAYSCALE)
                axes[1, i].imshow(gt, cmap="gray")
                axes[1, i].set_title("GT", fontsize=10)
            axes[1, i].axis("off")

        plt.tight_layout()
        plt.show()

for scene in SCENES:
    visualize_predictions(PRED_PATH, scene)

In [ ]:
# (선택) 커스텀 텍스트 쿼리 - Ramen 씬 예시 프롬프트로 초기화
import torch
import sys
sys.path.insert(0, ".")

from utils.vlm_utils import ClipSimMeasure
from nag_data import SemanticNAG
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder

QUERY_SCENE = SCENES[0]   # "ramen"
QUERY_TEXT = "egg"        # <-- 원하는 텍스트로 변경
# 라멘 씬 프롬프트 예시: egg, nori, chopsticks, wavy noodles in bowl, pork belly, kamaboko, sake cup, bowl, spoon, glass of water, napkin

nag_path = f"output/{SAVE_FOLDER}/{QUERY_SCENE}/sai_nag.pt"
if os.path.exists(nag_path):
    nag_data = torch.load(nag_path)
    snag = SemanticNAG(nag_data["nag"], nag_data["nag_feat"])

    vlm = ClipSimMeasure()
    vlm.load_model()
    vlm.encode_text(QUERY_TEXT)

    sims = [vlm.compute_similarity(f) for f in snag.feat]
    related = snag.get_related_gaussian(sims, topk=3, level=[2, 3])
    n_selected = (related > 0).sum().item()
    total = related.shape[0]

    print(f"Scene: {QUERY_SCENE}")
    print(f"Query: '{QUERY_TEXT}'")
    print(f"관련 가우시안: {n_selected} / {total} ({100 * n_selected / total:.1f}%)")
else:
    print(f"{nag_path} 없음 - Cell 3 파이프라인을 먼저 실행하세요")

---
## Cell 6. 6-Level 쿼리 실험 (Limitation 검증)

이 섹션은 **우리 자체 실험** 섹션이다. 참고: `md/ramen_experiment_plan.md`.

- **Level 0**: GT 카테고리 단어 그대로 (13, 정량)
- **Level 1**: 색상/재질/수량/외형 속성 추가 (13, 정량 — L0 GT 재사용)
- **Level 2**: 세부 파트 (14, 정성)
- **Level 3**: 인스턴스 구분 (4, 정성)
- **Level 4**: 공간 관계 (13, 정량 — L0 GT 재사용 + 관계무시 overlap 측정)
- **Level 5**: 설명문 (13, 정량 — L0 GT 재사용)

총 70개 쿼리 × 7 프레임.

In [ ]:
# [Cell 31] 쿼리 세트 정의 (Level 0~5, 70개)
# L1/L4/L5는 `gt_category` 필드로 L0 GT를 재사용한다.

EVAL_FRAMES = [
    "frame_00006", "frame_00024", "frame_00060", "frame_00065",
    "frame_00081", "frame_00119", "frame_00128",
]

QUERIES = {
    # Level 0: GT 카테고리 단어 그대로
    "L0": [
        {"q": "bowl",           "gt": "bowl"},
        {"q": "chopsticks",     "gt": "chopsticks"},
        {"q": "corn",           "gt": "corn"},
        {"q": "egg",            "gt": "egg"},
        {"q": "eggs",           "gt": "egg"},          # 단/복수 CLIP 민감도
        {"q": "glass of water", "gt": "glass of water"},
        {"q": "hand",           "gt": "hand"},
        {"q": "kamaboko",       "gt": "kamaboko"},
        {"q": "nori",           "gt": "nori"},
        {"q": "plate",          "gt": "plate"},
        {"q": "sake cup",       "gt": "sake cup"},
        {"q": "spoon",          "gt": "spoon"},
        {"q": "wavy noodles",   "gt": "wavy noodles"},
    ],
    # Level 1: 속성/어순 변화 (GT 재사용)
    "L1": [
        {"q": "yellow bowl",         "gt": "bowl"},
        {"q": "wooden chopsticks",   "gt": "chopsticks"},
        {"q": "yellow corn",         "gt": "corn"},
        {"q": "halved egg",          "gt": "egg"},
        {"q": "two eggs",            "gt": "egg"},
        {"q": "water glass",         "gt": "glass of water"},
        {"q": "person hand",         "gt": "hand"},
        {"q": "pink kamaboko",       "gt": "kamaboko"},
        {"q": "dark nori",           "gt": "nori"},
        {"q": "metal plate",         "gt": "plate"},
        {"q": "metal cup",           "gt": "sake cup"},
        {"q": "black spoon",         "gt": "spoon"},
        {"q": "curly noodles",       "gt": "wavy noodles"},
    ],
    # Level 2: 파트 (GT 없음, 정성)
    "L2": [
        {"q": "bowl rim",       "gt": None},
        {"q": "chopstick tip",  "gt": None},
        {"q": "egg yolk",       "gt": None},
        {"q": "egg white",      "gt": None},
        {"q": "glass rim",      "gt": None},
        {"q": "fingertips",     "gt": None},
        {"q": "kamaboko swirl", "gt": None},
        {"q": "nori edge",      "gt": None},
        {"q": "plate rim",      "gt": None},
        {"q": "cup rim",        "gt": None},
        {"q": "spoon handle",   "gt": None},
        {"q": "noodle strand",  "gt": None},
        {"q": "water",          "gt": None},
        {"q": "bottle",         "gt": None},
    ],
    # Level 3: 인스턴스 구분 (GT 없음, 정성 + left/right overlap)
    "L3": [
        {"q": "left chopstick", "gt": None, "pair": "right chopstick"},
        {"q": "right chopstick","gt": None, "pair": "left chopstick"},
        {"q": "left egg",       "gt": None, "pair": "right egg"},
        {"q": "right egg",      "gt": None, "pair": "left egg"},
        {"q": "both eggs",      "gt": None},
        {"q": "left nori",      "gt": None, "pair": "right nori"},
        {"q": "right nori",     "gt": None, "pair": "left nori"},
    ],
    # Level 4: 공간 관계 (L0 GT 재사용 + 관계무시 overlap)
    "L4": [
        {"q": "bowl on plate",          "gt": "bowl"},
        {"q": "chopsticks beside bowl", "gt": "chopsticks"},
        {"q": "corn near egg",          "gt": "corn"},
        {"q": "egg above noodles",      "gt": "egg"},
        {"q": "eggs above noodles",     "gt": "egg"},
        {"q": "glass beside bowl",      "gt": "glass of water"},
        {"q": "hand behind bowl",       "gt": "hand"},
        {"q": "kamaboko below eggs",    "gt": "kamaboko"},
        {"q": "nori beside eggs",       "gt": "nori"},
        {"q": "plate under bowl",       "gt": "plate"},
        {"q": "cup near chopsticks",    "gt": "sake cup"},
        {"q": "spoon behind bowl",      "gt": "spoon"},
        {"q": "noodles below eggs",     "gt": "wavy noodles"},
    ],
    # Level 5: 설명문 (L0 GT 재사용, 긴 prompt)
    "L5": [
        {"q": "An object containing noodles and various toppings on a table with sloping edges.", "gt": "bowl"},
        {"q": "A pair of utensils for holding food next to a yellow bowl.", "gt": "chopsticks"},
        {"q": "Corn kernels placed in a bowl, blended with the ingredients, floating on the surface.", "gt": "corn"},
        {"q": "The half of the egg next to the seaweed with the white evenly wrapped around the orange yolk.", "gt": "egg"},
        {"q": "Slices cut evenly in half down the center, above the ramen noodles.", "gt": "egg"},
        {"q": "A container with water resting on wooden tabletop and near wall.", "gt": "glass of water"},
        {"q": "Presented in a light grip and displaying the natural texture of the knuckles is a hand.", "gt": "hand"},
        {"q": "Smooth slices with a delicate spiral pattern featuring a swirl pattern.", "gt": "kamaboko"},
        {"q": "Long, dark green seaweed flakes placed against the side of the bowl.", "gt": "nori"},
        {"q": "A metal utensil that is convenient for holding cutlery, located on a wooden table and below the yellow bowl.", "gt": "plate"},
        {"q": "A drinking utensils with a smooth surface near a sake bottle.", "gt": "sake cup"},
        {"q": "The long one on the tray.", "gt": "spoon"},
        {"q": "Slender strips of food in the center of the bowl, mixed with toppings.", "gt": "wavy noodles"},
    ],
}

total = sum(len(v) for v in QUERIES.values())
print(f"총 쿼리 수: {total}")
for lvl, items in QUERIES.items():
    print(f"  {lvl}: {len(items)} queries")
print(f"평가 프레임: {EVAL_FRAMES}")

In [ ]:
# [Cell 32] 배치 예측 실행 — (프레임, 쿼리) 모든 쌍에 대해 mask 생성
# test_lerf.py의 핵심 경로를 인라인으로 구현. 랜덤 seed 고정.
import os, json, sys, time
import numpy as np
import torch
import cv2
from pathlib import Path

sys.path.insert(0, ".")
from omegaconf import OmegaConf
from scene import Scene, GaussianModel
from arguments import ModelParams, PipelineParams, OptimizationParams, get_combined_args
from gaussian_renderer import render
from utils.general_utils import safe_state
from utils.vlm_utils import ClipSimMeasure
from nag_data import SemanticNAG

safe_state(True)
torch.manual_seed(42); np.random.seed(42)

cfg = OmegaConf.load(CONFIG_FILE)
SAVE_FOLDER = cfg.dataset.save_folder
DATA_PATH = cfg.dataset.data_path

SCENE_NAME = SCENES[0]  # "ramen"
SOURCE_PATH = os.path.abspath(f"{DATA_PATH}/{SCENE_NAME}")
MODEL_PATH  = os.path.abspath(f"output/{SAVE_FOLDER}/{SCENE_NAME}")
EXP_OUT = f"output/render/lerf_exp/{SCENE_NAME}"
os.makedirs(EXP_OUT, exist_ok=True)

# --- Scene + Gaussian 로드 (딱 한 번) ---
class _Args:
    sh_degree = 3; embedding_dim = 10; feature_dim = 512; tab_len = 300
    source_path = SOURCE_PATH; model_path = MODEL_PATH
    images = "images"; resolution = -1; white_background = False
    data_device = "cuda"; eval = False
    render_items = ['RGB','Alpha','Normal','Depth','Edge','Curvature']
    convert_SHs_python = False; compute_cov3D_python = False
    depth_ratio = 0.0; debug = False

model_args = _Args()
gaussians = GaussianModel(model_args.sh_degree, 20)
scene = Scene(model_args, gaussians, 30000, load_sem=False)
bg = torch.tensor([0,0,0], dtype=torch.float32, device="cuda")

# sai_nag 로드
nag = torch.load(os.path.join(MODEL_PATH, "sai_nag.pt"))
snag = SemanticNAG(nag["nag"], nag["nag_feat"])
vlm = ClipSimMeasure(); vlm.load_model()

# 평가 프레임 이름 → 카메라 객체 매핑
train_cams = scene.getTrainCameras()
cam_by_name = {c.image_name: c for c in train_cams}
for fr in EVAL_FRAMES:
    assert fr in cam_by_name, f"{fr} 가 train cameras에 없음"

# GT polygon → binary mask 변환
def polygon_to_mask(h, w, points):
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [np.asarray(points, dtype=np.int32)], 1)
    return mask

def load_gt_masks(frame_name):
    """해당 프레임의 GT JSON → {category: binary_mask(H×W)} (동일 카테고리 여러 polygon은 합집합)."""
    jpath = f"{DATA_PATH}/label/{SCENE_NAME}/{frame_name}.json"
    if not os.path.exists(jpath):
        return {}
    anno = json.load(open(jpath))
    W, H = anno["info"]["width"], anno["info"]["height"]
    masks = {}
    for obj in anno["objects"]:
        c = obj["category"]
        m = polygon_to_mask(H, W, obj["segmentation"])
        masks[c] = np.maximum(masks.get(c, np.zeros((H, W), dtype=np.uint8)), m)
    return masks

# 캐시: (frame, gt_category) → binary mask
gt_cache = {f: load_gt_masks(f) for f in EVAL_FRAMES}

@torch.no_grad()
def predict_mask(cam, prompt, topk=3, levels=(2,3)):
    vlm.encode_text(prompt)
    sims = [vlm.compute_similarity(f) for f in snag.feat]
    point_valid = snag.get_related_gaussian(sims, topk=topk, level=list(levels))
    point_valid = point_valid.expand(-1, 20).cuda()
    gaussians._semantics = point_valid
    embd = render(cam, gaussians, model_args, bg)["semantics"]
    h, w = cam.image_height, cam.image_width
    m = (embd.reshape(20, -1)[0] > 0.5).reshape(h, w)
    return m.cpu().numpy().astype(np.uint8)

# ---- 실행 ----
t0 = time.time()
records = []
for level, items in QUERIES.items():
    lvl_dir_name = level.lower()
    for it in items:
        prompt = it["q"]
        gt_cat = it.get("gt")
        safe_qname = prompt.replace(" ", "_").replace(".", "").replace(",", "")[:80]
        for frame_name in EVAL_FRAMES:
            cam = cam_by_name[frame_name]
            out_dir = os.path.join(EXP_OUT, frame_name)
            os.makedirs(out_dir, exist_ok=True)

            # 예측
            pred = predict_mask(cam, prompt)
            pred_path = os.path.join(out_dir, f"{level}_{safe_qname}.png")
            cv2.imwrite(pred_path, pred * 255)

            # GT (있으면)
            gt = None
            if gt_cat is not None and gt_cat in gt_cache[frame_name]:
                gt = gt_cache[frame_name][gt_cat]
                gt_path = os.path.join(out_dir, f"{level}_{safe_qname}_gt.png")
                cv2.imwrite(gt_path, gt * 255)

            records.append({
                "level": level, "query": prompt, "gt_category": gt_cat,
                "frame": frame_name, "pred_path": pred_path,
                "has_gt": gt is not None,
                "pred_area": int(pred.sum()),
                "gt_area": int(gt.sum()) if gt is not None else 0,
            })
print(f"\n배치 예측 완료. {len(records)} 쌍, elapsed={time.time()-t0:.1f}s")
print(f"저장 위치: {EXP_OUT}/<frame>/<level>_<query>.png")

# records를 다음 셀에서 쓰도록 전역 보관
PRED_RECORDS = records

In [ ]:
# [Cell 33] 정량 평가 — L0/L1/L4/L5 IoU, L4 관계무시 overlap, L3 left/right overlap
import pandas as pd
from collections import defaultdict

def bin_metrics(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    tn = np.logical_and(~pred, ~gt).sum()
    eps = 1e-9
    iou  = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    return dict(IoU=iou, P=prec, R=rec, F1=f1, Acc=acc,
                pred_area=int(pred.sum()), gt_area=int(gt.sum()))

def read_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return (m > 128).astype(np.uint8) if m is not None else None

# records 재조합: (level, query, frame) → pred path
pred_lookup = {(r["level"], r["query"], r["frame"]): r["pred_path"] for r in PRED_RECORDS}

rows = []
for r in PRED_RECORDS:
    lvl, q, frm = r["level"], r["query"], r["frame"]
    pred = read_mask(r["pred_path"])
    if pred is None: continue

    entry = dict(level=lvl, query=q, frame=frm,
                 gt_category=r["gt_category"],
                 IoU=np.nan, P=np.nan, R=np.nan, F1=np.nan, Acc=np.nan,
                 overlap_vs_L0=np.nan)

    # (A) L0/L1/L4/L5 → L0 GT mask로 IoU
    if r["has_gt"] and r["gt_category"] is not None:
        gt = gt_cache[frm].get(r["gt_category"])
        if gt is not None and gt.shape == pred.shape:
            m = bin_metrics(pred, gt)
            entry.update(m)

    # (B) L4 (그리고 L1, L5도 함께) → L0 동일 gt_category 쿼리의 prediction과 overlap 측정
    if lvl in ("L1","L4","L5") and r["gt_category"] is not None:
        # L0에서 동일 gt_category를 가진 query 찾기 (첫 번째 매칭)
        l0_match = None
        for it in QUERIES["L0"]:
            if it["gt"] == r["gt_category"]:
                l0_match = it["q"]; break
        if l0_match is not None:
            l0_pred_path = pred_lookup.get(("L0", l0_match, frm))
            if l0_pred_path is not None:
                l0_pred = read_mask(l0_pred_path)
                if l0_pred is not None and l0_pred.shape == pred.shape:
                    inter = np.logical_and(pred.astype(bool), l0_pred.astype(bool)).sum()
                    union = np.logical_or (pred.astype(bool), l0_pred.astype(bool)).sum()
                    entry["overlap_vs_L0"] = inter / (union + 1e-9)

    rows.append(entry)

df = pd.DataFrame(rows)

# (C) L3 left/right pair overlap (instance 분리 실패 측정)
pair_rows = []
for r in PRED_RECORDS:
    if r["level"] != "L3": continue
    it = next(x for x in QUERIES["L3"] if x["q"] == r["query"])
    pair_q = it.get("pair")
    if pair_q is None: continue
    p1 = read_mask(r["pred_path"])
    p2_path = pred_lookup.get(("L3", pair_q, r["frame"]))
    if p2_path is None: continue
    p2 = read_mask(p2_path)
    if p1 is None or p2 is None or p1.shape != p2.shape: continue
    inter = np.logical_and(p1.astype(bool), p2.astype(bool)).sum()
    union = np.logical_or (p1.astype(bool), p2.astype(bool)).sum()
    pair_rows.append(dict(frame=r["frame"], q=r["query"], pair=pair_q,
                          iou=inter/(union+1e-9),
                          p1_area=int(p1.sum()), p2_area=int(p2.sum())))
pair_df = pd.DataFrame(pair_rows)

# 저장
csv_path = f"{EXP_OUT}/metrics.csv"; pair_csv = f"{EXP_OUT}/l3_pairs.csv"
df.to_csv(csv_path, index=False); pair_df.to_csv(pair_csv, index=False)
print(f"saved: {csv_path}, {pair_csv}")

# ---- Level별 mIoU 요약 ----
print("\n=== Level별 mIoU 요약 (정량 가능 쌍만) ===")
summary = df[df["IoU"].notna()].groupby("level").agg(
    mIoU=("IoU","mean"), mP=("P","mean"), mR=("R","mean"), mF1=("F1","mean"),
    n_pairs=("IoU","count")
).round(4)
print(summary)

# L4 관계 무시 정도
print("\n=== L1 / L4 / L5 : L0 예측과의 평균 overlap IoU (관계·속성·문장이 얼마나 L0 예측과 같은가) ===")
ov = df[df["overlap_vs_L0"].notna()].groupby("level")["overlap_vs_L0"].agg(["mean","median","count"]).round(4)
print(ov)

# L3 instance 분리
if len(pair_df) > 0:
    print("\n=== L3 left/right 페어 overlap (높을수록 instance 분리 실패) ===")
    print(pair_df.groupby("q")["iou"].agg(["mean","count"]).round(4))

### Level별 시각화 & mIoU — 6-Level Limitation 실험

THGS 파이프라인의 한계를 **쿼리 난이도를 6단계로 올려가며** 측정한다.

| Level | 무엇을 테스트하나 | GT | 저격 가설 | 핵심 지표 |
|---|---|---|---|---|
| L0 | baseline retrieval | O | 재현성 | L0 mIoU |
| L1 | 색상/재질/수량/어순 | O (L0 재사용) | CLIP attribute | L0-L1 drop, L1-L0 overlap |
| L2 | 파트 분할 (rim, yolk) | **X** | H2 SAM granularity | 정성 관찰만 |
| L3 | 좌/우 인스턴스 | **X** | H10 no-instance-id | pair overlap (> 0.8 = 실패) |
| L4 | 공간 관계 (on/beside) | O (L0 재사용) | **H10 관계 무시** | IoU(pred_L4, pred_L0) |
| L5 | 서술문 (long sentence) | O (L0 재사용) | CLIP description | L0-L5 drop |

**시각화 규약** (모든 Level 공통):
- **RGB overlay**: 원본 이미지 + 예측(빨강) + GT 있으면 초록
- **mask on black**: 검은 배경에 예측 흰색, GT 경계 초록 outline

**셀 구조**:
- Level당 독립 시각화 셀 → 결과 보면서 순서대로 진행 가능
- GT 없는 L2/L3는 mIoU 셀 생략 (정성 평가만)
- 마지막 종합 리포트 셀이 Level 간 비교 차트 + 텍스트 요약

아래 먼저 공통 헬퍼를 정의한 뒤 Level별 셀로 이동.

In [ ]:
# [Cell] 시각화 공통 헬퍼
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# PRED_RECORDS, df, pair_df, gt_cache, QUERIES, EVAL_FRAMES, EXP_OUT,
# DATA_PATH, SCENE_NAME 은 앞 셀에서 정의됨.

def _load_frame_image(frame_name):
    for root in [f"{DATA_PATH}/{SCENE_NAME}/images", f"{DATA_PATH}/label/{SCENE_NAME}"]:
        p = os.path.join(root, f"{frame_name}.jpg")
        if os.path.exists(p):
            return cv2.imread(p)
    return None

frame_imgs = {f: _load_frame_image(f) for f in EVAL_FRAMES}

def _read_mask(path):
    if not os.path.exists(path):
        return None
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return (m > 128).astype(np.uint8) if m is not None else None

def _overlay_rgb(img_bgr, mask, color_bgr=(0, 0, 255), alpha=0.45):
    out = img_bgr.copy()
    if mask is not None and mask.sum() > 0:
        colored = np.zeros_like(out)
        colored[mask.astype(bool)] = color_bgr
        out = cv2.addWeighted(out, 1.0, colored, alpha, 0)
    return out

def _mask_on_black(shape_hw, pred_mask, gt_mask=None):
    canvas = np.zeros((shape_hw[0], shape_hw[1], 3), dtype=np.uint8)
    if pred_mask is not None and pred_mask.sum() > 0:
        canvas[pred_mask.astype(bool)] = (255, 255, 255)
    if gt_mask is not None and gt_mask.sum() > 0:
        contours, _ = cv2.findContours(gt_mask.astype(np.uint8),
                                        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(canvas, contours, -1, (0, 255, 0), 2)
    return canvas

def _safe_name(prompt):
    return prompt.replace(" ", "_").replace(".", "").replace(",", "")[:80]

def visualize_level(level_name, show=True, save=True):
    '''Level별 시각화: 각 (query, frame) -> [RGB overlay | mask on black] 2 panel.'''
    items = QUERIES[level_name]
    rows = len(items)
    cols = len(EVAL_FRAMES) * 2
    fig, axes = plt.subplots(rows, cols, figsize=(1.6 * cols, 2.0 * rows))
    if rows == 1:
        axes = axes[None, :]
    fig.suptitle(
        f"Level {level_name} — Ramen  |  각 frame: [RGB overlay | mask-on-black]",
        fontsize=12,
    )
    for i, it in enumerate(items):
        prompt = it["q"]
        gt_cat = it.get("gt")
        for j, frm in enumerate(EVAL_FRAMES):
            pred_p = os.path.join(EXP_OUT, frm, f"{level_name}_{_safe_name(prompt)}.png")
            pred = _read_mask(pred_p)
            img = frame_imgs[frm]
            gt = gt_cache[frm].get(gt_cat) if gt_cat else None

            vis_rgb = _overlay_rgb(img, pred, (0, 0, 255), 0.45)
            if gt is not None:
                vis_rgb = _overlay_rgb(vis_rgb, gt, (0, 255, 0), 0.30)
            axes[i, j * 2].imshow(cv2.cvtColor(vis_rgb, cv2.COLOR_BGR2RGB))

            mob = _mask_on_black(img.shape[:2], pred, gt)
            axes[i, j * 2 + 1].imshow(cv2.cvtColor(mob, cv2.COLOR_BGR2RGB))

            for ax in (axes[i, j * 2], axes[i, j * 2 + 1]):
                ax.set_xticks([])
                ax.set_yticks([])
            if i == 0:
                axes[i, j * 2].set_title(frm.replace("frame_", "f"), fontsize=8)
                axes[i, j * 2 + 1].set_title("mask", fontsize=8)
            if j == 0:
                label = prompt[:35] + ("…" if len(prompt) > 35 else "")
                axes[i, 0].set_ylabel(
                    label, fontsize=8, rotation=0, ha="right", va="center", labelpad=90
                )
    plt.tight_layout()
    if save:
        out_path = os.path.join(EXP_OUT, f"viz_{level_name}.png")
        plt.savefig(out_path, dpi=100, bbox_inches="tight")
        print(f"saved: {out_path}")
    if show:
        plt.show()
    else:
        plt.close()


def miou_for_level(level_name):
    '''GT 있는 Level만 per-query mIoU + 전체 mIoU 출력.'''
    d = df[(df["level"] == level_name) & df["IoU"].notna()]
    if len(d) == 0:
        print(f"[{level_name}] 정량 가능 데이터 없음 (GT 없음).")
        return None
    per_q = d.groupby("query").agg(
        mIoU=("IoU", "mean"), mP=("P", "mean"), mR=("R", "mean"),
        mF1=("F1", "mean"), n=("IoU", "count"),
    ).round(4).sort_values("mIoU", ascending=False)
    print(f"=== Level {level_name} — per-query mIoU (n={len(d)} 쌍) ===")
    print(per_q.to_string())
    print()
    print(f"[{level_name}] 전체 평균 mIoU = {d['IoU'].mean():.4f}  "
          f"(P={d['P'].mean():.4f}, R={d['R'].mean():.4f})")
    return per_q


print("헬퍼 로드 완료. visualize_level('L0') / miou_for_level('L0') 호출 가능.")


#### Level L0 — Base Retrieval (시각화)

**실험 목적**: 저자 파이프라인의 **baseline 성능 재현**.

- GT LERF-OVS 카테고리 단어를 그대로 쿼리로 사용 (`bowl`, `chopsticks`, `egg`, ...)
- 총 13개 쿼리 × 7 프레임 = 91 (query, frame) 쌍
- GT polygon이 실제로 존재하는 쌍만 정량 평가 가능 (약 59쌍)

**측정**: 각 쿼리의 예측 마스크가 GT polygon과 얼마나 겹치는가 (IoU).

**의미**: 이 mIoU가 **이후 Level들의 기준점**이 된다. 
- 저자 논문의 LERF-OVS ramen mIoU와 비교해 환경 재현성 검증
- L1/L4/L5의 drop은 이 값 대비로만 의미있음

**빨강** = 예측, **초록** = GT polygon (mask-on-black은 GT 경계만 초록선).

In [ ]:
visualize_level('L0')

#### Level L0 — mIoU

**baseline mIoU**. 저자 논문의 LERF-OVS ramen 수치와 비교해 재현 여부 판단.
per-query 테이블은 어떤 카테고리가 쉬운지/어려운지 보여준다.

- 높은 mIoU 예상: `bowl`, `egg`, `chopsticks`, `wavy noodles` (크고 단일 객체)
- 낮은 mIoU 예상: `hand` (1 프레임만), `spoon` (2 프레임만), `glass of water` (투명)

In [ ]:
miou_for_level('L0')

#### Level L1 — 속성·어순 변화 (시각화)

**실험 목적**: CLIP의 **attribute grounding 능력** 측정. 
"training-free VLM이 색상/재질/수량 같은 속성을 얼마나 이해하는가?"

- L0 쿼리에 속성을 덧붙여 재질문
  - 색상: `yellow bowl`, `yellow corn`, `pink kamaboko`, `dark nori`, `black spoon`
  - 재질: `wooden chopsticks`, `metal plate`, `metal cup`
  - 수량: `two eggs`
  - 외형: `halved egg`, `curly noodles`
  - 어순: `water glass` (L0는 `glass of water`)
  - 소유자: `person hand`
- **GT는 L0과 동일하게 재사용** (속성이 추가돼도 같은 객체이므로)

**저격 가설**: H3 (CLIP object-centric bias), H10 (bag-of-masks).

**해석 방법**:
- L1 mIoU가 L0과 비슷 → 속성을 무시하고 주 명사만 봄 (= CLIP robustness 부족)
- L1 mIoU가 훨씬 낮음 → 속성이 오히려 노이즈가 돼서 매칭 실패

**추가 지표**: 종합 리포트에 `IoU(pred_L1, pred_L0)` histogram이 함께 나옴. 0.85 초과 비율이 높으면 "속성이 예측에 영향 없음"의 증거.

In [ ]:
visualize_level('L1')

#### Level L1 — mIoU

**속성 robustness 측정**. L0과 같은 GT 대비 IoU이므로 **L0 mIoU와 직접 비교 가능**.

- L0 대비 **drop이 작다** (≤5) → 속성이 무시됨 = CLIP이 주 명사만 봄
- L0 대비 **drop이 크다** (>10) → 속성이 오히려 노이즈로 작용해서 매칭 실패

종합 리포트에서 `IoU(pred_L1, pred_L0)` histogram과 함께 보면 어느 방향인지 확실해진다.

In [ ]:
miou_for_level('L1')

#### Level L2 — 세부 파트 (시각화)

**실험 목적**: THGS의 **파트 레벨 분할 능력** 측정.
"슈퍼포인트 계층이 객체를 파트 단위까지 쪼개는가?"

- 객체 일부분을 지칭하는 쿼리 14개
  - 경계: `bowl rim`, `plate rim`, `glass rim`, `cup rim`, `nori edge`
  - 작은 구조: `chopstick tip`, `fingertips`, `spoon handle`, `noodle strand`
  - 내부 구성: `egg yolk`, `egg white`, `kamaboko swirl`, `water`, `bottle`
- **GT 없음** (LERF-OVS는 part-level GT 제공 안 함) → 정성 평가만

**저격 가설**: H2 (SAM multi-scale granularity gap), H9 (fixed hierarchy schedule).

**해석 방법**:
- 대부분 실패할 것으로 예상. 저자 파이프라인은 `level=1` (SAM s-scale) 단일 사용이라 yolk/white 같은 미세 구조의 mask가 애초에 없을 가능성 큼.
- `bowl rim` → bowl 전체가 나옴 / `egg yolk` → egg 전체가 나옴 같은 **coarse fallback** 패턴 관찰.

**mIoU 없음**: 정량은 불가. 시각화 결과에서 RGB overlay와 mask-on-black을 눈으로 보며 GOOD / PARTIAL / WRONG을 판단.

In [ ]:
visualize_level('L2')

#### Level L3 — 인스턴스 구분 (시각화)

**실험 목적**: THGS가 **같은 카테고리의 여러 인스턴스를 구별**할 수 있는가?
"좌/우 젓가락, 좌/우 계란을 따로 집을 수 있는가?"

- 7개 쿼리 (3개 pair + 1 all): `left chopstick`, `right chopstick`, `left egg`, `right egg`, `both eggs`, `left nori`, `right nori`
- **GT 없음** → pair overlap으로 간접 측정

**저격 가설**: H10의 결정적 증거.
- THGS는 카테고리 단위 feature만 가지고 있고 **instance ID가 없다**
- 슈퍼포인트가 유사 feature로 뭉쳐 있어 좌/우가 같은 feature일 가능성 큼

**해석 방법**:
- `left chopstick` 예측 vs `right chopstick` 예측이 **거의 동일**하면 instance 분리 실패
- 종합 리포트 4번째 패널에 `left/right pair overlap IoU` 막대가 표시됨
- 0.8 이상이면 **"같은 것을 고른다"** = instance 분리 실패의 정량 증거

**mIoU 없음**: GT 없음. pair overlap이 그 자리를 대체.

In [ ]:
visualize_level('L3')

#### Level L4 — 공간 관계 (시각화)

**실험 목적**: CLIP text encoder가 **공간 관계어**(on/above/beside/near/behind/below/under)를 이해해 다른 객체를 집는가?

- L0 주체 객체에 관계 수식어를 붙임
  - `bowl on plate`, `chopsticks beside bowl`, `egg above noodles`
  - `kamaboko below eggs`, `plate under bowl`, `noodles below eggs`, `hand behind bowl`
- **GT는 L0 주체 객체와 동일**. 관계 쿼리가 L0과 다른 객체를 집는다면 IoU가 오히려 낮게 나와야 올바른 이해.

**저격 가설**: **H10의 가장 강력한 증거**.
- CLIP은 bag-of-words 성격이라 "bowl on plate"와 "bowl"의 embedding이 거의 같음
- 결과적으로 **관계는 무시되고 주 명사만 반영**된 예측이 나올 것

**두 가지 지표**:
1. **L0 GT 대비 IoU**: L0과 비슷하면 "주체는 맞게 집음"
2. **`IoU(pred_L4, pred_L0)`** (L4 mIoU 셀에서 자동 출력): L4 예측이 L0 예측과 얼마나 같은가
   - `>0.85` 비율이 높을수록 = 공간관계어가 예측에 거의 영향 없음 = **관계 무시의 결정적 증거**
   - 이 수치가 이 실험의 **핵심 결론 숫자**

In [ ]:
visualize_level('L4')

#### Level L4 — mIoU + 관계 무시 정도

**두 가지 숫자가 함께 출력된다**:

1. **L4 mIoU** (L0 GT 대비): 관계 쿼리가 **주체 객체를 잡았는지**. 
   - L0과 비슷한 수준 → 관계 무시하고 주체 객체만 집음
   - 훨씬 낮음 → 아예 다른 영역을 예측 (관계어가 noise로 작용)

2. **`IoU(pred_L4, pred_L0)`** — **이 실험의 핵심 결과**:
   - `>0.85` 비율이 높을수록 = L4 예측이 L0 예측과 거의 동일
   - = "공간관계어가 예측에 거의 영향 없다" = **H10의 결정적 증거**
   - 이 한 수치가 "THGS는 공간 이해 못한다"를 정량으로 증명.

In [ ]:
miou_for_level('L4')

# L4 '관계 무시' 정도 측정
_ov = df[(df.level == 'L4') & df.overlap_vs_L0.notna()]['overlap_vs_L0']
if len(_ov):
    print()
    print(f"[L4] 관계 무시 정도 — IoU(pred_L4, pred_L0) "
          f"평균={_ov.mean():.4f}, median={_ov.median():.4f}")
    print(f"    >0.85 비율 = {(_ov > 0.85).sum()}/{len(_ov)} "
          f"쌍 ({(_ov > 0.85).mean():.2%}) — CLIP이 공간관계를 무시하는 증거")


#### Level L5 — 설명문 (시각화)

**실험 목적**: CLIP이 **긴 referring-expression 문장**을 얼마나 이해하는가?
"외형/위치/기능으로 서술한 문장만으로 객체를 찾을 수 있는가?"

- 13개 L0 카테고리에 대응하는 **서술문 (Ref-LERF 스타일)**
  - 예: `bowl` → "An object containing noodles and various toppings on a table with sloping edges."
  - 예: `spoon` → "The long one on the tray."
  - 예: `chopsticks` → "A pair of utensils for holding food next to a yellow bowl."
- **GT는 L0과 동일** (서술하는 대상이 같은 객체)

**저격 가설**: H3 (CLIP description robustness).
- CLIP text encoder는 긴 문장에서 token들이 평균화되어 feature가 흐려짐
- 단어 단위 쿼리보다 성능이 떨어질 것으로 예상

**해석 방법**:
- **L0 대비 IoU drop** 크기가 CLIP의 문장 이해 약점의 크기
- drop이 작으면: CLIP이 설명문에서도 핵심 명사를 잘 집음
- drop이 크면: 부가 설명이 오히려 노이즈가 돼서 매칭 붕괴
- `IoU(pred_L5, pred_L0)` histogram이 높은 구간에 몰리면 = "긴 문장도 결국 핵심 명사 때문에 같은 결과를 냄" (robustness의 역설)

In [ ]:
visualize_level('L5')

#### Level L5 — mIoU

**설명문 robustness 측정**. L0과 같은 GT 대비 IoU.

- L0 대비 drop이 클수록 → CLIP이 긴 문장에서 feature 희석을 겪음
- drop이 거의 없으면 → 문장이 길어도 핵심 명사가 여전히 지배적 (robustness 있으나 문장 이해는 아니라는 뜻)

L5가 L0과 비슷하게 나오면 의미있는 해석: **"training-free VLM은 사실상 단어 기반으로만 동작한다"**.

In [ ]:
miou_for_level('L5')

### 종합 리포트 — Level 간 비교

6개 Level 실험을 다 돌린 뒤 **하나의 figure + 텍스트 요약**으로 결론을 뽑는다.

4-패널 차트:
1. **Level별 평균 mIoU bar** — L0/L1/L4/L5의 mIoU 막대.
2. **per-(category, frame) L0 vs 다른 Level scatter** — 대각선 아래 = L0보다 나쁜 쌍. Level별로 분포가 어떻게 다른지.
3. **예측 overlap histogram** — `IoU(pred_X, pred_L0)`. 임계 0.85 초과 비율이 높을수록 해당 변형(속성/관계/문장)이 예측에 영향 없음 = 한계의 증거.
4. **L3 left/right pair overlap** — instance 분리 실패도.

텍스트 요약에는:
- Level별 평균 mIoU
- L0 대비 {L1, L4, L5}의 drop (mean/median/worst)
- L1/L4/L5 예측이 L0과 같을 확률
- L3 인스턴스 분리 성공률

이 셀의 출력이 **논문 결론 프레임의 1페이지 요약**이 된다.

In [ ]:
# [Cell 종합] Level 간 비교 차트 + 텍스트 요약
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(16, 10))

# (1) Level별 mIoU bar (GT 있는 Level만)
ax1 = fig.add_subplot(2, 2, 1)
sub = (
    df[df["IoU"].notna()]
    .groupby("level")["IoU"].mean()
    .reindex(["L0", "L1", "L4", "L5"]).dropna()
)
colors = ["#4C72B0", "#DD8452", "#55A467", "#C44E52"][: len(sub)]
ax1.bar(sub.index, sub.values, color=colors)
for i, v in enumerate(sub.values):
    ax1.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)
ax1.set_title("Level별 평균 IoU (GT 있는 Level만)")
ax1.set_ylabel("mIoU")
ax1.set_ylim(0, max(0.6, sub.max() * 1.2) if len(sub) else 1.0)

# (2) per-(category, frame) L0 vs 다른 Level IoU scatter
ax2 = fig.add_subplot(2, 2, 2)
pivot = df[df["IoU"].notna()].pivot_table(
    index=["gt_category", "frame"], columns="level", values="IoU"
)
for lvl, color in zip(["L1", "L4", "L5"], ["#DD8452", "#55A467", "#C44E52"]):
    if "L0" in pivot.columns and lvl in pivot.columns:
        x = pivot["L0"].values
        y = pivot[lvl].values
        m = ~(np.isnan(x) | np.isnan(y))
        ax2.scatter(x[m], y[m], alpha=0.6, label=lvl, color=color, s=40)
ax2.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
ax2.set_xlabel("L0 IoU")
ax2.set_ylabel("L1/L4/L5 IoU")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.legend()
ax2.set_title("L0 대비 각 Level IoU (대각선 아래 = L0 보다 나쁨)")

# (3) 관계·속성·문장 무시 histogram
ax3 = fig.add_subplot(2, 2, 3)
for lvl, color in zip(["L1", "L4", "L5"], ["#DD8452", "#55A467", "#C44E52"]):
    d = df[(df["level"] == lvl) & df["overlap_vs_L0"].notna()]
    ax3.hist(
        d["overlap_vs_L0"].values, bins=np.linspace(0, 1, 21),
        alpha=0.55, label=f"{lvl} (n={len(d)})", color=color,
    )
ax3.axvline(0.85, color="red", linestyle=":", lw=1)
ax3.text(0.86, ax3.get_ylim()[1] * 0.9, "임계=0.85", color="red", fontsize=8)
ax3.set_xlabel("IoU(pred_X, pred_L0)")
ax3.set_ylabel("count")
ax3.set_title("예측이 L0과 얼마나 동일한가 (높을수록 변형 무시)")
ax3.legend()

# (4) L3 pair overlap
ax4 = fig.add_subplot(2, 2, 4)
if len(pair_df) > 0:
    grp = pair_df.groupby("q")["iou"].mean().sort_values(ascending=False)
    ax4.barh(grp.index, grp.values, color="#8172B3")
    for i, v in enumerate(grp.values):
        ax4.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
    ax4.axvline(0.8, color="red", linestyle=":", lw=1)
    ax4.set_xlim(0, 1.05)
    ax4.set_title("L3 left/right pair overlap (높을수록 instance 분리 실패)")
else:
    ax4.text(0.5, 0.5, "L3 pair 데이터 없음", ha="center", va="center")

plt.tight_layout()
summary_path = os.path.join(EXP_OUT, "summary_report.png")
plt.savefig(summary_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"saved: {summary_path}")

# 텍스트 리포트
print()
print("=" * 70)
print("종합 텍스트 리포트")
print("=" * 70)
print()
print("[Level별 평균 mIoU]")
print(sub.round(4).to_string())

if "L0" in pivot.columns:
    print()
    print("[L0 대비 IoU drop]")
    for lvl in ["L1", "L4", "L5"]:
        if lvl in pivot.columns:
            drop = (pivot["L0"] - pivot[lvl]).dropna()
            print(f"  L0 → {lvl}: mean={drop.mean():+.4f}  "
                  f"median={drop.median():+.4f}  worst={drop.max():+.4f}")

print()
print("[L1/L4/L5 vs L0 예측 overlap]")
for lvl in ["L1", "L4", "L5"]:
    d = df[(df.level == lvl) & df.overlap_vs_L0.notna()]["overlap_vs_L0"]
    if len(d):
        print(f"  {lvl}: mean={d.mean():.4f}  median={d.median():.4f}  "
              f">0.85 비율={(d > 0.85).mean():.2%}")

if len(pair_df) > 0:
    high = (pair_df["iou"] > 0.8).mean()
    print()
    print("[L3 instance 분리]")
    print(f"  평균 left/right overlap = {pair_df['iou'].mean():.4f}")
    print(f"  >0.8 비율 = {high:.2%} (instance 분리 실패)")

print()
print("L2 (파트)는 viz_L2.png 참조. GT 없으므로 mIoU 없음.")
print("L3 (인스턴스)도 GT 없으므로 mIoU 없음. pair overlap이 그 자리를 대체.")
